<a href="https://colab.research.google.com/github/satjot3-rgb/Supply-Chain-Data-Engineering-Pipeline/blob/main/Supply_Chain_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Environment Setup and Creating Spark Session in Google Colab

In [ ]:
# 1. Deep Clean: Remove all previous Spark installations to prevent conflicts
print("Cleaning up previous Spark versions...")
!rm -rf spark-*

# 2. Environment Setup
print("Installing Java 11 and Spark 3.5.4...")
!apt-get update -qq > /dev/null
!apt-get install openjdk-11-jdk-headless -qq > /dev/null
!wget -q https://archive.apache.org/dist/spark/spark-3.5.4/spark-3.5.4-bin-hadoop3.tgz
!tar xf spark-3.5.4-bin-hadoop3.tgz

# 3. Install Python Dependencies
print("Installing PySpark and Findspark...")
!pip install -q pyspark==3.5.4 findspark

import os
import sys

# 4. Set Environment Variables
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.4-bin-hadoop3"
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

# 5. Initialize Findspark and clean path
import findspark
findspark.init()

from pyspark.sql import SparkSession

try:
    # Force stop any stale gateway
    spark_check = SparkSession.getActiveSession()
    if spark_check:
        spark_check.stop()

    # 6. Re-create Spark Session with exact version-matched Delta JAR
    spark = SparkSession.builder \
        .appName("DataCoSupplyChain") \
        .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.2.0") \
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
        .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
        .getOrCreate()

    print(f"Spark Session Created Successfully!")
    print(f"Spark Version: {spark.version}")

except Exception as e:
    print(f"Failed to start Spark: {e}")
    print("\nIMPORTANT: If the 'JavaPackage' error remains, please click 'Runtime' -> 'Restart Session' at the top of the page, then run this cell once more.")

# Bronze Layer

**Load Dataset**

In [ ]:
raw_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("DataCoSupplyChainDataset.csv")

print("Rows:", raw_df.count())
print("Columns:", len(raw_df.columns))

**Store Bronze Table**

In [ ]:
# Enable Column Mapping to allow spaces and special characters in Delta table column names
raw_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("delta.columnMapping.mode", "name") \
    .option("delta.minReaderVersion", "2") \
    .option("delta.minWriterVersion", "5") \
    .save("/mnt/bronze/dataco_raw")

# Silver Layer

**Load Bronze and Duplicates Check**

In [ ]:
from pyspark.sql.functions import *

# Load from Bronze
df = spark.read.format("delta").load("/mnt/bronze/dataco_raw")

duplicate_count = df.count() - df.dropDuplicates().count()
print("Duplicate Rows:", duplicate_count)

**Null Analysis**

In [ ]:
null_report = df.select([
    count(
        when(col(c).isNull(), c)
    ).alias(c)
    for c in df.columns
])

null_report.show(truncate=False)

**Data Cleaning**

In [ ]:
clean_df = df.dropDuplicates()

clean_df = clean_df.fillna({
    "Sales":0,
    "Order Profit Per Order":0,
    "Benefit per order":0
})

print(clean_df.count())

**Rename Important Columns**

In [ ]:
clean_df = (
    clean_df
    .withColumnRenamed("Order Id","order_id")
    .withColumnRenamed("Customer Id","customer_id")
    .withColumnRenamed("Product Card Id","product_id")
    .withColumnRenamed("Product Name","product_name")
    .withColumnRenamed("Category Name","category_name")
    .withColumnRenamed("Sales","sales_amount")
    .withColumnRenamed(
        "Order Profit Per Order",
        "profit_per_order"
    )
)

**Date Transformations**

In [ ]:
clean_df = (
    clean_df
    .withColumn(
        "order_date",
        to_timestamp(
            col("order date (DateOrders)")
        )
    )
    .withColumn(
        "order_year",
        year("order_date")
    )
    .withColumn(
        "order_month",
        month("order_date")
    )
)

**Delivery Delay KPI**

In [ ]:
clean_df = (
    clean_df
    .withColumn(
        "delivery_delay_days",
        col("Days for shipping (real)")
        - col("Days for shipment (scheduled)")
    )
)

clean_df.select(
    "delivery_delay_days"
).show()

Store Silver table

In [43]:
clean_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("delta.columnMapping.mode", "name") \
    .option("delta.minReaderVersion", "2") \
    .option("delta.minWriterVersion", "5") \
    .save("/mnt/silver/orders")

# Gold Layer

**Monthly Revenue Analysis**

In [ ]:
monthly_sales = (
    clean_df
    .groupBy(
        "order_year",
        "order_month"
    )
    .agg(
        round(
            sum("sales_amount"),
            2
        ).alias("revenue")
    )
    .orderBy(
        "order_year",
        "order_month"
    )
)

monthly_sales.show()

**Category Performance**

In [ ]:
category_sales = (
    clean_df
    .groupBy("category_name")
    .agg(
        round(
            sum("sales_amount"),
            2
        ).alias("revenue"),

        round(
            sum("profit_per_order"),
            2
        ).alias("profit")
    )
    .orderBy(
        desc("revenue")
    )
)

category_sales.show()

**Shipping Performance**

In [ ]:
shipping_perf = (
    clean_df
    .groupBy("Shipping Mode")
    .agg(
        round(
            avg("delivery_delay_days"),
            2
        ).alias("avg_delay")
    )
)

shipping_perf.show()

**Top Customers**

In [ ]:
top_customers = (
    clean_df
    .groupBy("customer_id")
    .agg(
        round(
            sum("sales_amount"),
            2
        ).alias("total_sales")
    )
    .orderBy(
        desc("total_sales")
    )
)

top_customers.show(10)

**Fraud Analysis**

In [ ]:
fraud_orders = (
    clean_df
    .filter(
        col("Order Status")
        == "SUSPECTED_FRAUD"
    )
)

print(
    "Fraud Orders:",
    fraud_orders.count()
)

**Window Function**

In [ ]:
from pyspark.sql.window import Window

monthly_product_sales = (
    clean_df
    .groupBy(
        "product_id",
        "order_year",
        "order_month"
    )
    .agg(
        sum("sales_amount")
        .alias("sales")
    )
)
window_spec = Window \
    .partitionBy("product_id") \
    .orderBy(
        "order_year",
        "order_month"
    )

growth_df = (
    monthly_product_sales
    .withColumn(
        "previous_sales",
        lag("sales",1)
        .over(window_spec)
    )
)
growth_df = (
    growth_df
    .withColumn(
        "growth_pct",
        round(
            (
                col("sales")
                - col("previous_sales")
            )
            /
            col("previous_sales")
            * 100,
            2
        )
    )
)

growth_df.show()

**Export Gold Tables**

In [44]:
monthly_sales.toPandas().to_csv(
    "monthly_sales.csv",
    index=False
)

category_sales.toPandas().to_csv(
    "category_sales.csv",
    index=False
)

shipping_perf.toPandas().to_csv(
    "shipping_performance.csv",
    index=False
)

top_customers.toPandas().to_csv(
    "top_customers.csv",
    index=False
)